신입생 충원현황 계열확장 완료된 파일과 원래 분류 되어있는 등록금 파일의  공통 (기준년도, 학교, 대계열) 조합 수: 6,080개, 공통 학교 수: 202개
그에 맞춰 모든 연도에 존재하는 공통 조합, 공통학교에 해당하는 컬럼 데이터만 남기고 두 피처 파일을 병합함.
그에 따라 원래 설립 구분, 지역 등이 없던 등록금 데이터도 신입생 충원현황 데이터 옆에 컬럼이 되어 구분이 가능해짐.

<업데이트 사항>
수도권/지방 구분 그에 따른  충원율_평균, 충원율_편차, 충원율_표준편차, 충원안정성비율을 지역대신 수도권/지방을 이용하여 계산한 데이터를 뒤에 _수도권을 붙인 컬럼을 생성 후 데이터를 넣어줌
0값의 경우 :	해당 그룹 내 모든 학교가 동일한 충원율을 가짐 or 그룹에 학교 1개여서 그렇게 진행됨.

+ 재학생 충원현황에 대한 피처 추가 및 파생 피처 계산  추가
+ 1인당 장학금 피처 추가


In [ ]:
# 1. 필요한 라이브러리 불러오기
import pandas as pd


In [ ]:
# 2. 파일 불러오기
df_enrollment = pd.read_csv("/content/신입생_충원_현황(2014~2023)_충원율피처_정정본.csv")
df_register = pd.read_csv("/content/학과별_등록금_최종.csv")


In [ ]:
#  3. 연도별 공통 (학교, 대계열) 조합 구하기
school_major_per_year_enr = df_enrollment.groupby('기준년도')[['학교', '대계열']].apply(lambda x: set(tuple(row) for row in x.to_numpy()))
common_school_major_enr = set(school_major_per_year_enr.iloc[0])
for s in school_major_per_year_enr[1:]:
    common_school_major_enr &= s

school_major_per_year_reg = df_register.groupby('기준연도')[['학교명', '대계열']].apply(lambda x: set(tuple(row) for row in x.to_numpy()))
common_school_major_reg = set(school_major_per_year_reg.iloc[0])
for s in school_major_per_year_reg[1:]:
    common_school_major_reg &= s

common_school_major_both = common_school_major_enr & common_school_major_reg


In [ ]:
#  4. 신입생 충원율 데이터에서 공통 조합 필터링 및 정렬
df_enrollment['조합'] = list(zip(df_enrollment['학교'], df_enrollment['대계열']))
df_enrollment_common = df_enrollment[df_enrollment['조합'].isin(common_school_major_both)]
df_enrollment_common_sorted = df_enrollment_common.sort_values(by=['기준년도', '학교']).reset_index(drop=True)


In [ ]:
#  5. 등록금 데이터에서 공통 조합 필터링 및 평균 등록금 추출
df_register['조합'] = list(zip(df_register['학교명'], df_register['대계열']))
df_register_common = df_register[df_register['조합'].isin(common_school_major_both)]
등록금_매핑 = df_register_common.groupby(['학교명', '대계열'])['등록금'].mean().reset_index()
등록금_매핑.columns = ['학교', '대계열', '등록금']


In [ ]:
#  6. 등록금 컬럼 병합
df_final = df_enrollment_common_sorted.merge(등록금_매핑, on=['학교', '대계열'], how='left')
df_final = df_final.drop(columns=['조합'])  # 중간 컬럼 제거


In [ ]:
#  단일 그룹일 경우 파생 피처를 NaN으로 처리 (프로젝트 상 0값으로 채우는 것 보다는 NAN값으로 채우는 것이 맞음)
group_counts = df_final.groupby(['기준년도', '지역', '설립구분', '대계열'])['학교'].transform('count')
df_final.loc[group_counts == 1, ['충원율_표준편차', '충원안정성비율']] = float('nan')

In [ ]:
# 7. 저장
df_final.to_csv("/content/신입생_충원현황_공통조합_등록금추가.csv", index=False, encoding='utf-8-sig')
df_final.head()


In [ ]:
# 수도권/비수도권 구분 생성
df['수도권구분'] = df['지역'].apply(lambda x: '수도권' if x in ['서울', '경기', '인천'] else '비수도권')

In [ ]:
# 수도권 기준 파생 피처 계산
group_cols_capital = ['기준년도', '수도권구분', '설립구분', '대계열']
df['충원율_평균_수도권기준'] = df.groupby(group_cols_capital)['충원율(%)'].transform('mean')
df['충원율_편차_수도권기준'] = df['충원율(%)'] - df['충원율_평균_수도권기준']
df['충원율_표준편차_수도권기준'] = df.groupby(group_cols_capital)['충원율(%)'].transform('std')
df['충원안정성비율_수도권기준'] = df['충원율_표준편차_수도권기준'] / df['충원율_평균_수도권기준']

In [ ]:
# 단일 그룹 NaN 처리
group_counts_capital = df.groupby(group_cols_capital)['학교'].transform('count')
df.loc[group_counts_capital == 1, [
    '충원율_표준편차_수도권기준',
    '충원안정성비율_수도권기준'
]] = float('nan')

In [ ]:
# 파일 저장
df.to_csv("신입생_충원현황_공통조합_등록금추가_수도권기준포함.csv", index=False, encoding='utf-8-sig')

In [ ]:
# 재학생 충원현황 파일에 피처들 병합 하기     
#라이브러리 로드 및 데이터 불러오기
import pandas as pd

df_fresh = pd.read_csv("/content/신입생_충원현황_공통조합_등록금추가_수도권기준포함.csv")
df_re = pd.read_csv("/content/학과별_재학생충원율_최종.csv")

#  컬럼명 통일
df_re = df_re.rename(columns={
    '기준연도': '기준년도',
    '학교명': '학교'
})

# 기준: 기준년도 + 학교 + 대계열
df_merged = df_fresh.merge(
    df_re[['기준년도', '학교', '대계열', '재학생', '실모집인원', '재학생충원율']],
    on=['기준년도', '학교', '대계열'],
    how='left'
)

#  병합 결과 저장
df_merged.to_csv("/content/신입생+재학생_충원현황_통합.csv", index=False, encoding='utf-8-sig')
df_merged.head()


FileNotFoundError: [Errno 2] No such file or directory: '/content/신입생_충원현황_공통조합_등록금추가_수도권기준포함.csv'

In [ ]:
#재학생 충원현황에 있는 피처와 지역, 수도권/지방을 이용한 파생 피처 계산
#라이브러리 로드 및 데이터 불러오기
import pandas as pd

df = pd.read_csv("/content/신입생+재학생_충원현황_통합.csv")

#  수도권/비수도권 구분 컬럼 생성
df['수도권구분'] = df['지역'].apply(lambda x: '수도권' if x in ['서울', '경기', '인천'] else '비수도권')

# 지역 기준 재학생 충원율 파생 피처 계산
group_cols_region = ['기준년도', '지역', '설립구분', '대계열']
df['재학생충원율_평균'] = df.groupby(group_cols_region)['재학생충원율'].transform('mean')
df['재학생충원율_편차'] = df['재학생충원율'] - df['재학생충원율_평균']
df['재학생충원율_표준편차'] = df.groupby(group_cols_region)['재학생충원율'].transform('std')
df['재학생충원안정성비율'] = df['재학생충원율_표준편차'] / df['재학생충원율_평균']

# 단일 그룹 NaN 처리
group_counts_region = df.groupby(group_cols_region)['학교'].transform('count')
df.loc[group_counts_region == 1, [
    '재학생충원율_표준편차',
    '재학생충원안정성비율'
]] = float('nan')

#  수도권 기준 재학생 충원율 파생 피처 계산
group_cols_capital = ['기준년도', '수도권구분', '설립구분', '대계열']
df['재학생충원율_평균_수도권기준'] = df.groupby(group_cols_capital)['재학생충원율'].transform('mean')
df['재학생충원율_편차_수도권기준'] = df['재학생충원율'] - df['재학생충원율_평균_수도권기준']
df['재학생충원율_표준편차_수도권기준'] = df.groupby(group_cols_capital)['재학생충원율'].transform('std')
df['재학생충원안정성비율_수도권기준'] = df['재학생충원율_표준편차_수도권기준'] / df['재학생충원율_평균_수도권기준']

# 단일 그룹 NaN 처리
group_counts_capital = df.groupby(group_cols_capital)['학교'].transform('count')
df.loc[group_counts_capital == 1, [
    '재학생충원율_표준편차_수도권기준',
    '재학생충원안정성비율_수도권기준'
]] = float('nan')

# 결과 저장
df.to_csv("/content/신입생+재학생_충원현황_통합_재학생파생포함.csv", index=False, encoding='utf-8-sig')
df.head()


# 1인당 장학금 피처 추가
import pandas as pd

df_base = pd.read_csv("/content/신입생+재학생_충원현황_통합_재학생파생포함.csv")
df_scholar = pd.read_csv("/content/학과별_장학금_최종.csv")

# 컬럼명 통일
df_scholar = df_scholar.rename(columns={
    '기준연도': '기준년도',
    '학교명': '학교'
})

# 기준: 기준년도 + 학교 + 대계열
df_merged = df_base.merge(
    df_scholar[['기준년도', '학교', '대계열', '1인당장학금']],
    on=['기준년도', '학교', '대계열'],
    how='left'
)

#  결과 저장
df_merged.to_csv("/content/신입생+재학생_충원현황_장학금포함.csv", index=False, encoding='utf-8-sig')
df_merged.head()
